# Example 3 — Equity-Aware Capital Allocation

This notebook compares an efficiency-only allocation strategy with equity-aware research scenarios under a fixed synthetic capital budget.


In [ ]:
import pandas as pd
import plotly.express as px

from equitable_capital import (
    allocate_capital,
    generate_synthetic_startups,
    summarize_allocation,
    train_model,
)


In [ ]:
data = generate_synthetic_startups(n=1500, seed=42)
result = train_model(data)
scored = result.scored_data
budget = 5_000_000


## Compare the baseline with several equity weights


In [ ]:
baseline, _ = allocate_capital(scored, budget=budget, equity_weight=0.0)
rows = [
    {
        "scenario": "Efficiency-only",
        "equity_weight": 0.0,
        **summarize_allocation(baseline, budget),
    }
]

for weight in [0.15, 0.30, 0.45, 0.60]:
    _, equitable = allocate_capital(
        scored, budget=budget, equity_weight=weight
    )
    rows.append(
        {
            "scenario": f"Equity-aware {weight:.2f}",
            "equity_weight": weight,
            **summarize_allocation(equitable, budget),
        }
    )

summary = pd.DataFrame(rows)
summary


## Visualize the allocation trade-off


In [ ]:
fig = px.scatter(
    summary,
    x="share_to_higher_barrier_contexts",
    y="expected_successes",
    size="businesses_funded",
    color="equity_weight",
    hover_name="scenario",
    title="Synthetic Allocation Trade-off",
    labels={
        "share_to_higher_barrier_contexts": "Share to Higher-Barrier Contexts",
        "expected_successes": "Expected Successes",
    },
)
fig.show()


## Inspect one equity-aware allocation


In [ ]:
_, equitable_030 = allocate_capital(
    scored, budget=budget, equity_weight=0.30
)
equitable_030[[
    "startup_id",
    "state",
    "industry",
    "requested_capital",
    "allocated_capital",
    "predicted_success_probability",
    "underserved_context_index",
]].head(20)


The simulator is a research prototype. The selected equity weight is a scenario parameter, not a policy recommendation or real-world lending rule.
